In [ ]:
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7
)

tip_ids = [4, 8, 12, 16, 20]

def detect_gesture(lm_list):
    fingers = []

    # Thumb
    if lm_list[tip_ids[0]][1] > lm_list[tip_ids[0]-1][1]:
        fingers.append(1)
    else:
        fingers.append(0)

    # Other fingers
    for i in range(1,5):
        if lm_list[tip_ids[i]][2] < lm_list[tip_ids[i]-2][2]:
            fingers.append(1)
        else:
            fingers.append(0)

    if fingers == [0,0,0,0,0]:
        return "FIST"

    elif fingers == [1,1,1,1,1]:
        return "OPEN PALM"

    elif fingers == [1,0,0,0,0]:
        return "THUMBS UP"

    elif fingers == [0,1,1,0,0]:
        return "PEACE"

    elif fingers == [0,1,0,0,0]:
        return "ONE"

    elif fingers == [0,1,1,0,0]:
        return "TWO"

    elif fingers == [0,1,1,1,0]:
        return "THREE"

    elif fingers == [1,1,0,0,1]:
        return "OK"

    else:
        return "UNKNOWN"


cap = cv2.VideoCapture(0)

while True:
    success, img = cap.read()
    img = cv2.flip(img,1)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    gesture = "None"

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:

            lm_list = []
            for id, lm in enumerate(hand_landmarks.landmark):
                h,w,c = img.shape
                cx,cy = int(lm.x*w), int(lm.y*h)
                lm_list.append([id,cx,cy])

            if len(lm_list)!=0:
                gesture = detect_gesture(lm_list)

            mp_draw.draw_landmarks(img, hand_landmarks, mp_hands.HAND_CONNECTIONS)

    cv2.putText(img, f'Gesture: {gesture}', (10,70),
                cv2.FONT_HERSHEY_SIMPLEX,1.5,(0,255,0),3)

    cv2.imshow("Hand Gesture Recognition", img)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
out.write(frame)